# 文档加载器 Document Loaders（基本快过时了，重点看 docling）
## 1 加载 txt

In [1]:
from langchain_community.document_loaders import TextLoader

# 定义TextLoader对象，file_path=".txt的位置"
text_loader = TextLoader(file_path="../asset/load/01-langchain-utf-8.txt", encoding="utf-8")

# 加载
docs = text_loader.load() #返回List列表(Document对象)

print(docs)

[Document(metadata={'source': '../asset/load/01-langchain-utf-8.txt'}, page_content='LangChain 是一个用于构建基于大语言模型（LLM）应用的开发框架，旨在帮助开发者更高效地集成、管理和增强大语言模型的能力，构建端到端的应用程序。它提供了一套模块化工具和接口，支持从简单的文本生成到复杂的多步骤推理任务')]


Document 对象中有两个重要的属性：
- page_content：真正的文档内容，字符串类型。
- metadata：文档内容的元数据，字典类型。

In [ ]:
print(type(docs[0])) #langchain_core.documents.base.Document
print(docs[0].page_content)
print(docs[0].metadata) # {'source': './data/langchain.txt'}

In [2]:
from langchain_community.document_loaders import TextLoader

# 定义TextLoader对象，file_path=".txt的位置"
text_loader = TextLoader(file_path="../asset/load/01-langchain-gbk.txt", encoding="gbk")

# 加载
docs = text_loader.load() #返回List列表(Document对象)

print(docs)

[Document(metadata={'source': '../asset/load/01-langchain-gbk.txt'}, page_content='LangChain 是一个用于构建基于大语言模型（LLM）应用的开发框架，旨在帮助开发者更高效地集成、管理和增强大语言模型的能力，构建端到端的应用程序。它提供了一套模块化工具和接口，支持从简单的文本生成到复杂的多步骤推理任务')]


## 2 加载 csv

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(file_path="asset/load/04-load.csv")

data = loader.load()

print(data)
print(type(data)) # <class 'list'>
print(type(data[0])) # <class 'langchain_core.documents.base.Document'>
print(len(data)) # 4
print(data[0].page_content) # id: 1 title: Introduction to Python ...

## 3 加载 JSON

In [ ]:
from langchain_community.document_loaders import JSONLoader
from rich import print as rprint

# 定义JSONLoader对象
# 情况1
json_loader=JSONLoader(
    file_path="../asset/load/03-load.json",
    jq_schema=".", ## 提取所有字段
    text_content=False # 是否提取字符串？当提取字段是字符串时设置为 True，False 的话则保持原始 JSON 结构，将提取的数据转换为JSON字符串存入 page_content字段中
)
# 情况2
# .messages[].content:遍历.messages[]中所有元素 从每一个元素中提取.content字段
# json_loader=JSONLoader(
# file_path="../asset/load/03-load.json",
# jq_schema=".messages[].content"
# )

docs = json_loader.load()

rprint(docs)

In [ ]:
from langchain_community.document_loaders import JSONLoader
from rich import print as rprint

# 定义json文件的路径
file_path = '../asset/load/03-response.json'

# 定义JSONLoader对象
# 需求1：提取data.items中的数据
# loader = JSONLoader(
#     file_path=file_path, # 文件路径
#     jq_schema=".data.items[]",
#     text_content=False, # 提取内容是否为字符串格式
# )

# 需求2：提取data.items[].content中的数据
# loader = JSONLoader(
#     file_path=file_path, # 文件路径
#     jq_schema=".data.items[].content",
# )

# 需求3：提取data.items中指定字段的数据
loader = JSONLoader(
    file_path=file_path, # 文件路径
    jq_schema="""
        .data.items[] | {
            author,
            created_at,
            content: (.title + "\n" + .content)
        }
    """,
    text_content=False, # 提取内容是否为字符串格式
)

data = loader.load()
rprint(data)

4 加载 PDF
### 方式1 PyPDFLoader

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    # 文件路径，支持本地文件和在线文件链接
    # file_path="../asset/load/04-sample.pdf",
    file_path="https://arxiv.org/pdf/alg-geom/9202012",
    # 提取模式:控制如何从 PDF 文件中解析和提取文本结构。
    # plain 提取文本，默认值
    # layout 布局感知提取模式，通常会通过插入大量的空格、换行符，来模拟原文档中的多栏、 缩进和间距（适用场景：学术论文（如 arXiv 论文）、多栏报刊杂志、带有左右分栏的合同）
    extraction_mode="plain",
)

docs = loader.load()

print(docs)
print(len(docs))

### 方式2 MinerU

In [ ]:
import os
import time
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

